# 04 - Evaluación del Modelo con MLflow

## Objetivo
Determinar el número óptimo de clústeres (K) evaluando:
1. **Método del Codo** → inercia (WCSS)
2. **Silhouette Score** → calidad de separación entre clústeres

Todos los experimentos quedan registrados en MLflow.
Fuente: `bigdata.default.gold_customer_features`

In [0]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import json
import warnings
warnings.filterwarnings("ignore")

df_gold = spark.table("bigdata.default.gold_customer_features")
print("Registros en Gold:", df_gold.count())
df_gold.show(5)

In [0]:
pdf = df_gold.select("id_cliente", "recency", "frequency", "monetary").toPandas()

print(f"Clientes únicos: {len(pdf):,}")
print("\nEstadísticas descriptivas:")
print(pdf[["recency", "frequency", "monetary"]].describe().round(2))

In [0]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(pdf[["recency", "frequency", "monetary"]])

print("Escalado completado.")
print(f"Media post-escalado  : {rfm_scaled.mean(axis=0).round(4)}")
print(f"StdDev post-escalado : {rfm_scaled.std(axis=0).round(4)}")

## Parte 3 — Evaluación K=2..8 con MLflow

Para cada K entrenamos K-Means y registramos en MLflow:
- Parámetro: K
- Métrica: Inercia (Método del Codo)
- Métrica: Silhouette Score
- Artefacto: modelo entrenado

| Silhouette Score | Interpretación |
|-----------------|----------------|
| ~1.0 | Clústeres bien separados y compactos |
| ~0.0 | Solapamiento entre clústeres |
| < 0  | Puntos asignados al clúster incorrecto |

In [0]:
mlflow.set_experiment("/Users/yefrye.avilaz@konradlorenz.edu.co/rfm_clustering_experiment")

ks = [2, 3, 4, 5, 6, 7, 8]
results = []

print("Evaluando K-Means para K =", ks)
print("-" * 55)

for k in ks:
    with mlflow.start_run(run_name=f"kmeans_k_{k}"):

        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(rfm_scaled)

        inertia    = km.inertia_
        silhouette = silhouette_score(rfm_scaled, labels)

        mlflow.log_param("k", k)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("n_init", 10)
        mlflow.log_metric("inertia", inertia)
        mlflow.log_metric("silhouette", silhouette)
        mlflow.sklearn.log_model(km, "kmeans_model")

        results.append({"K": k, "Inertia": inertia, "Silhouette": silhouette})
        print(f"K={k}  |  Inercia: {inertia:>10,.1f}  |  Silhouette: {silhouette:.4f}")

print("-" * 55)
print(f"✅ {len(results)} experimentos registrados en MLflow")

In [0]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## Parte 4 — Visualización: Método del Codo y Silhouette Score

- Codo: busca el punto donde la curva de inercia deja de bajar significativamente
- Silhouette: busca el K con el valor máximo

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(results_df["K"], results_df["Inertia"],
             marker="o", linewidth=2, markersize=9, color="#1f77b4")
axes[0].set_xlabel("Número de Clústeres (K)", fontsize=12)
axes[0].set_ylabel("Inercia (WCSS)", fontsize=12)
axes[0].set_title("Método del Codo", fontsize=14, fontweight="bold")
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(results_df["K"])

axes[1].plot(results_df["K"], results_df["Silhouette"],
             marker="o", linewidth=2, markersize=9, color="#2ca02c")
axes[1].set_xlabel("Número de Clústeres (K)", fontsize=12)
axes[1].set_ylabel("Silhouette Score", fontsize=12)
axes[1].set_title("Silhouette Score por K", fontsize=14, fontweight="bold")
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(results_df["K"])

plt.suptitle("Evaluación de K-Means — Selección del K Óptimo",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Busca el codo en la izquierda y el máximo en la derecha.")

In [0]:
best_row        = results_df.loc[results_df["Silhouette"].idxmax()]
best_k          = int(best_row["K"])
best_silhouette = float(best_row["Silhouette"])
best_inertia    = float(best_row["Inertia"])

print("=" * 50)
print(f"  K ÓPTIMO SELECCIONADO : {best_k}")
print(f"  Silhouette Score      : {best_silhouette:.4f}")
print(f"  Inercia               : {best_inertia:,.1f}")
print("=" * 50)
print("\nRanking completo:")
print(results_df.sort_values("Silhouette", ascending=False).to_string(index=False))

In [0]:
k_config = {
    "optimal_k":        best_k,
    "silhouette_score": best_silhouette,
    "inertia":          best_inertia
}

with open("/tmp/kmeans_config.json", "w") as f:
    json.dump(k_config, f)

print(f"✅ K óptimo guardado en /tmp/kmeans_config.json → K={best_k}")
print("Ahora ejecuta el notebook 04 para entrenar el modelo final.")